# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and references all data entities by their `@id` as defined in the Croissant schema.

### Dataset Source
The dataset is defined by a Croissant schema accessible at the provided URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load the dataset metadata and preview details using `mlcroissant`. The `metadata` object provides high-level documentation about the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL for FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (note: this downloads metadata and discovers record sets, fields, etc)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets and the fields within each record set, referencing entities by their `@id`.

We'll enumerate all record sets with their `@id`, name, and the fields (with field `@id`s) they contain.

In [ ]:
# List available record sets and their fields by @id
record_set_ids = []
record_set_info = {}
for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs.id} | name: {rs.name}")
    field_ids = []
    for field in rs.fields:
        print(f"    Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
        field_ids.append(field.id)
    record_set_ids.append(rs.id)
    record_set_info[rs.id] = field_ids
    print("")
print(f"Total Record Sets found: {len(record_set_ids)}")

## 3. Data Extraction
Load records from each record set into pandas DataFrames. You can select a specific record set by its `@id` for EDA.
**Note:** You can explore DataFrames for each record set and their available columns.

In [ ]:
# Load each record set into a DataFrame (indexed by @id)
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"    Columns: {df.columns.tolist()}")
    print(f"    Shape: {df.shape}")
    print("")
# As an example, let's select the first available record set:
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print("First record set preview:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps—reference fields by `@id`. Here, we show how to filter, normalize, and group data using the relevant field IDs.
**Instructions:**
- Choose a *numeric* field `@id` for analysis. For demonstration, this will use the first record set and the first numeric field it contains.

In [ ]:
import numpy as np

# Helper to find a numeric field in selected record set
numeric_field_id = None
group_field_id = None
selected_record_set_id = main_record_set_id
if selected_record_set_id:
    fields = [f for f in metadata.get_record_set(selected_record_set_id).fields]
    # Find first numeric field
    for f in fields:
        if f.data_type in ("Float", "Number", "Integer"):
            numeric_field_id = f.id
            break
    # Find a possible group field (e.g., categorical, but not the numeric)
    for f in fields:
        if f.id != numeric_field_id and f.data_type in ("Text", "String"):
            group_field_id = f.id
            break

    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    df = dataframes[selected_record_set_id]
    if numeric_field_id in df.columns:
        # Convert to numeric (if not already)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = np.nanmean(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}):")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Groupby (if group_field found)
        if group_field_id and group_field_id in df.columns:
            grouped_df = (
                filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            )
            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
    else:
        print(f"No numeric field found in columns for EDA in this record set.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using Matplotlib/Seaborn for the selected numeric field and group field (if available).

You can customize field IDs below for other visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field found, boxplot by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(y=numeric_field_id, x=group_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization cannot be produced: required fields missing.")

## 6. Conclusion
We've demonstrated how to explore and analyze the FAIR^2 dataset using `mlcroissant`, referencing entities by their `@id` as per the Croissant schema. You can extend this notebook by refining the field selections and adding domain-specific visualizations and analysis for your research tasks.